# RQ2 functional-resource coverage audit

CPU-only diagnostic. Enumerates all endpoint-locked K=4 and K=5 anchor sets using frozen development geometry, then relates Uniform-to-FinalGeo coverage changes to seed-3/4 interim validation accuracy. It does not read checkpoints or the CIFAR-100 test split.

In [ ]:
import os, subprocess, sys, zipfile, json
from pathlib import Path
from kaggle_secrets import UserSecretsClient
github_token = UserSecretsClient().get_secret('github_token')
assert github_token, 'Missing Kaggle secret github_token'
PROJECT_ROOT = Path('/kaggle/working/new-pruning')
askpass = Path('/kaggle/working/.github_git_askpass.py')
askpass.write_text("#!/usr/bin/env python3\nimport os,sys\np=sys.argv[1] if len(sys.argv)>1 else ''\nprint('x-access-token' if 'Username' in p else os.environ['GITHUB_TOKEN_RUNTIME'])\n")
askpass.chmod(0o700)
env = os.environ.copy(); env.update({'GIT_ASKPASS':str(askpass),'GIT_TERMINAL_PROMPT':'0','GITHUB_TOKEN_RUNTIME':github_token})
try:
    command = ['git','-C',str(PROJECT_ROOT),'pull','--ff-only'] if (PROJECT_ROOT/'.git').is_dir() else ['git','clone','https://github.com/duyh80456-code/new-pruning.git',str(PROJECT_ROOT)]
    subprocess.run(command, env=env, check=True)
finally:
    askpass.unlink(missing_ok=True); github_token = None
os.chdir(PROJECT_ROOT); sys.path.insert(0, str(PROJECT_ROOT))
print('Commit:', subprocess.run(['git','rev-parse','HEAD'],capture_output=True,text=True,check=True).stdout.strip())

## Locate frozen development geometry and interim validation results

In [ ]:
import rq2_anchor_placement
RQ2_INPUT = Path('/kaggle/input/notebooks/dyhngg/test-rq2')
assert RQ2_INPUT.exists(), f'Attach RQ2-v1 output: {RQ2_INPUT}'
RQ2_ROOT = rq2_anchor_placement.find_rq2_development_root(RQ2_INPUT, '/kaggle/working/materialized-rq2-resource-audit')
interim_files = sorted(Path('/kaggle/input').rglob('interim_validation_dense_all.csv'))
assert len(interim_files) == 1, f'Attach exactly one interim-validation output; found: {interim_files}'
INTERIM_CSV = interim_files[0]
print('RQ2 development root:', RQ2_ROOT)
print('Interim validation metrics:', INTERIM_CSV)

## Run deterministic CPU audit

In [ ]:
from rq2_resource_coverage_audit import run_resource_coverage_audit
OUTPUT_DIR = Path('/kaggle/working/rq2-resource-coverage-audit')
result = run_resource_coverage_audit(RQ2_ROOT, INTERIM_CSV, OUTPUT_DIR)
print(json.dumps(result, indent=2))

In [ ]:
import pandas as pd
from IPython.display import display, Image
k4 = pd.read_csv(OUTPUT_DIR/'rq2_k4_resource_geometry_candidates.csv')
k5 = pd.read_csv(OUTPUT_DIR/'rq2_k5_resource_geometry_candidates.csv')
widthwise = pd.read_csv(OUTPUT_DIR/'rq2_widthwise_resource_geometry_accuracy.csv')
print('Uniform, PureGeo, FinalGeo:')
display(k4[k4.reference_set.notna() & k4.reference_set.ne('')][['reference_set','anchors','R_G','R_res','mean_resource_distance','max_gap','compute_ratio_vs_uniform_K4','functional_resource_pareto']])
print('Widths where FinalGeo worsens resource coverage:')
display(widthwise[widthwise.resource_coverage_worse_in_finalgeo])
print('Best K=5 candidates satisfying R_G < Uniform and R_res <= Uniform:')
display(k5[k5.joint_improvement_feasible].sort_values(['R_G','compute_ratio_vs_uniform_K4']).head(20))
display(Image(filename=str(OUTPUT_DIR/'rq2_k4_functional_resource_pareto.png')))
display(Image(filename=str(OUTPUT_DIR/'rq2_widthwise_resource_coverage_accuracy.png')))

In [ ]:
import shutil
archive = shutil.make_archive('/kaggle/working/rq2-resource-coverage-audit', 'zip', root_dir=OUTPUT_DIR)
print('Download/persist:', archive)